# Step 1: Install Required Libraries

Install all required libraries for embeddings, vector database and Google Gemini.

In [1]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q google-generativeai
!pip install -q python-dotenv

# Step 2: Import Required Libraries

Import all Python libraries required for the RAG pipeline.

In [3]:
import os
import pickle

import numpy as np
import pandas as pd

import faiss

from sentence_transformers import SentenceTransformer

from dotenv import load_dotenv

import google.generativeai as genai

# Step 3: Load Resume Dataset

Load the final processed resume dataset created in the previous notebook.

In [4]:
df = pd.read_csv("../data/final_feature_dataset.csv")

print("Dataset Shape :", df.shape)

df.head()

Dataset Shape : (3500, 16)


,ResumeID,Category,Skills,Education,Experience,Clean_Text,Text,Source,Resume_Length,Word_Count,Sentence_Count,Skill_Count,Education_Length,Experience_Length,Source_Encoded,Category_Encoded
0,REAL_0001,Java Developer,"Python, SQL, Git, Linux",Computer Science degree,jessica claire montgomery street san francisco...,jessica claire montgomery street san francisco...,jessica claire montgomery street san francisco...,ResumeAtlas,1495,189,1,4,3,64,0,17
1,REAL_0002,Java Developer,"Python, SQL, Git, Linux",Computer Science degree,jared arthur maica java developer 17994568777 ...,jared arthur maica java developer linkedincomi...,jared arthur maica java developer 17994568777 ...,ResumeAtlas,1686,206,1,4,3,62,0,17
2,REAL_0003,Java Developer,"Python, SQL, Git, Linux",Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,jessica claire 9 resumesampleexamplecom montgo...,jessica claire 9 resumesampleexamplecom 555 43...,ResumeAtlas,5555,715,1,4,3,62,0,17
3,REAL_0004,Java Developer,"Python, SQL, Git, Linux",Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,jessica claire 9 resumesampleexamplecom montgo...,jessica claire 9 resumesampleexamplecom 555 43...,ResumeAtlas,12834,1657,1,4,3,61,0,17
4,REAL_0005,Java Developer,"Python, SQL, Git, Linux",Computer Science degree,jessica claire 100 montgomery st 10th floor xx...,jessica claire 100 montgomery st 10th floor xx...,jessica claire 100 montgomery st 10th floor xx...,ResumeAtlas,4181,489,1,4,3,57,0,17


# Step 4: Select a Resume

Select one resume that will be used for building the RAG pipeline.

In [5]:
resume = df.loc[0, "Clean_Text"]

print(resume[:1000])

jessica claire montgomery street san francisco ca resumesampleexamplecom professional summary highly skilled software development professional bringing 10 years software design development integration advanced knowledge java skills agile html xml jdbc tomcat work history senior java developertech lead 2014 current synnex corporation tracy ca java developer agile scrum team javascript java develop customer facing internal web applications underlying component applications wrote maintainable extensible code team environment implemented designs including experimentation multiple iterations system administrator mantech international corporation joint base mcguire nj technical lead system administrator oracle ebusiness suite erp application mentored technical staff java developerscada system admin big lots anniston al developed maintained internal web applications managed scada system wind energy business business analyst john deere financial city state gathered requirements small large pro

## Step 5: Split Resume into Chunks

Large documents are divided into smaller chunks before generating embeddings.

This improves semantic search performance because the embedding model captures the meaning of smaller text segments more effectively.

In [6]:
def split_resume(text, chunk_size=120):

    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):

        chunk = " ".join(words[i:i+chunk_size])

        chunks.append(chunk)

    return chunks

## Step 6: Generate Resume Chunks

Apply the chunking function and inspect the generated chunks.

In [8]:
resume_chunks = split_resume(resume)

print("Total Resume Chunks :", len(resume_chunks))

Total Resume Chunks : 2


## Step 7: Display Sample Chunks

Display the first two chunks to verify that the resume has been split correctly.

In [9]:
print("--------------- Chunk 1 ---------------")

print(resume_chunks[0])

print("\n")

print("--------------- Chunk 2 ---------------")

print(resume_chunks[1])

--------------- Chunk 1 ---------------
jessica claire montgomery street san francisco ca resumesampleexamplecom professional summary highly skilled software development professional bringing 10 years software design development integration advanced knowledge java skills agile html xml jdbc tomcat work history senior java developertech lead 2014 current synnex corporation tracy ca java developer agile scrum team javascript java develop customer facing internal web applications underlying component applications wrote maintainable extensible code team environment implemented designs including experimentation multiple iterations system administrator mantech international corporation joint base mcguire nj technical lead system administrator oracle ebusiness suite erp application mentored technical staff java developerscada system admin big lots anniston al developed maintained internal web applications managed scada system wind energy business business analyst john deere financial city sta

##  Step 8: Check Chunk Length

Verify the number of words present in each chunk.

In [11]:
for i, chunk in enumerate(resume_chunks):

    print(f"Chunk {i+1} :", len(chunk.split()), "words")

Chunk 1 : 120 words
Chunk 2 : 69 words


## Step 9: Load Sentence Transformer Model

Load a pre-trained Sentence Transformer model to convert each resume chunk into dense vector embeddings.

In [12]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Sentence Transformer Model Loaded Successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence Transformer Model Loaded Successfully


## Step 10: Generate Embeddings

Convert every resume chunk into a numerical vector representation using the Sentence Transformer model.

In [14]:
chunk_embeddings = embedding_model.encode(
    resume_chunks,
    convert_to_numpy=True
)

print("Embedding Shape :", chunk_embeddings.shape)

Embedding Shape : (2, 384)


## Step 11: Inspect Embeddings

Display the first few values of the first embedding vector.

In [15]:
print(chunk_embeddings[0][:10])

[-0.01126862 -0.034894    0.02269279 -0.01545688 -0.01620611  0.00116433
 -0.01117534  0.02051886 -0.13130404 -0.01814988]


## Step 12: Create FAISS Vector Database

Create a FAISS index and store all chunk embeddings for semantic retrieval.

In [16]:
import faiss

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(chunk_embeddings)

print("Total Vectors in FAISS :", index.ntotal)

Total Vectors in FAISS : 2


## Step 13: Create a Search Function

Create a function that converts a user query into an embedding and retrieves the most relevant resume chunks from the FAISS vector database.

In [17]:
def retrieve_chunks(query, top_k=2):

    # Convert query into embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    # Search similar chunks
    distances, indices = index.search(
        query_embedding,
        top_k
    )

    return distances, indices

## Step 14: Retrieve Relevant Resume Chunks

Retrieve the most relevant chunks for a sample user query.

In [18]:
query = "What programming languages does the candidate know?"

distances, indices = retrieve_chunks(query)

print("Distances :", distances)

print("Indices :", indices)

Distances : [[1.2976322 1.3864666]]
Indices : [[0 1]]


## Step 15: Display Retrieved Chunks

Display the retrieved resume chunks that are most relevant to the user query.

In [20]:
for idx in indices[0]:

    print("=" * 80)

    print(resume_chunks[idx])

    print("\n")

jessica claire montgomery street san francisco ca resumesampleexamplecom professional summary highly skilled software development professional bringing 10 years software design development integration advanced knowledge java skills agile html xml jdbc tomcat work history senior java developertech lead 2014 current synnex corporation tracy ca java developer agile scrum team javascript java develop customer facing internal web applications underlying component applications wrote maintainable extensible code team environment implemented designs including experimentation multiple iterations system administrator mantech international corporation joint base mcguire nj technical lead system administrator oracle ebusiness suite erp application mentored technical staff java developerscada system admin big lots anniston al developed maintained internal web applications managed scada system wind energy business business analyst john deere financial city state gathered requirements


small large p

## Step 16: Build Context for the Language Model

Combine the retrieved chunks into a single context that will later be provided to Gemini for answer generation.

In [22]:
context = "\n\n".join(
    [resume_chunks[i] for i in indices[0]]
)

print(context)

jessica claire montgomery street san francisco ca resumesampleexamplecom professional summary highly skilled software development professional bringing 10 years software design development integration advanced knowledge java skills agile html xml jdbc tomcat work history senior java developertech lead 2014 current synnex corporation tracy ca java developer agile scrum team javascript java develop customer facing internal web applications underlying component applications wrote maintainable extensible code team environment implemented designs including experimentation multiple iterations system administrator mantech international corporation joint base mcguire nj technical lead system administrator oracle ebusiness suite erp application mentored technical staff java developerscada system admin big lots anniston al developed maintained internal web applications managed scada system wind energy business business analyst john deere financial city state gathered requirements

small large pr

### Step 17: Load Gemini API

Load the Gemini API key from the environment file and configure the language model.

In [23]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.0-flash")

print("Gemini Configured Successfully")

Gemini Configured Successfully


In [25]:
faiss.write_index(
    index,
    "../models/resume_index.faiss"
)

print("FAISS Index Saved Successfully")

FAISS Index Saved Successfully


In [26]:
with open("../models/resume_chunks.pkl", "wb") as file:
    pickle.dump(resume_chunks, file)

print("Resume Chunks Saved Successfully")

Resume Chunks Saved Successfully


### Step 18: Generate AI Response

Provide the retrieved resume context and the user query to Gemini to generate an intelligent response.

In [3]:
prompt = f"""
You are an AI Resume Analyzer.

Resume Context:

{context}

Question:

{query}

Answer in simple English.
"""

response = model.generate_content(prompt)

print(response.text)

NameError: name 'retrieved_text' is not defined